In [1]:
import torch
import torch.nn as nn
import torchtext
from torch.utils.data import Dataset, DataLoader

In [2]:
DEVICE = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

print("Using PyTorch version: {}, Device: {}".format(torch.__version__, DEVICE))
print("Using torchtext version: {}".format(torchtext.__version__))

Using PyTorch version: 1.12.0, Device: cpu
Using torchtext version: 0.13.0


In [3]:
# ----- AG News Dataset -----

In [4]:
train_data, test_data = torchtext.datasets.AG_NEWS(root='./data')
labels = [_, 'World', 'Sports', 'Business', 'Sci/Tech']
y, x = next(iter(train_data))
print(labels[y])
print(x)

Business
Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\band of ultra-cynics, are seeing green again.


In [5]:
# label 종류
set([label for (label, text) in train_data]), set([label for (label, text) in test_data])

({1, 2, 3, 4}, {1, 2, 3, 4})

In [6]:
# ----- Text Data Preprocessing -----

In [9]:
from torchtext.data.utils import get_tokenizer

tokenizer = get_tokenizer('basic_english')
tokenizer("Hi, my name is Joonseok!") # 알파벳을 다 소문자로 바꿔줌

['hi', ',', 'my', 'name', 'is', 'joonseok', '!']

In [10]:
from torchtext.vocab import build_vocab_from_iterator

def tokens(data_iter):
    for _, text in data_iter:
        yield tokenizer(text)

# train_data에 있는 토큰들만을 대상으로 인덱스 생성하고 토큰들을 인덱스로 mappiong
encoder = build_vocab_from_iterator(tokens(train_data), specials=["<unk>"])
encoder.set_default_index(encoder["<unk>"]) 
encoder(tokenizer("Hi, my name is Joonseok <unk> !")) # train_data에 없는 토큰이 나오는 경우 인덱스를 0으로 반환

[24104, 3, 1300, 951, 21, 0, 0, 764]

In [11]:
text_pipeline = lambda x: encoder(tokenizer(x))
label_pipeline = lambda x: int(x) - 1 # 1, 2, 3, 4를 0, 1, 2, 3으로 mapping

In [10]:
# ----- Text Data Batch Preprocessing -----
# rnn은 어떠한 길이의 데이터도 다 처리 가능하지만, 
# rnn을 배치 단위로 처리하기 위해서는 배치 안에 있는 텍스트끼리는 길이가 같아야 함

In [12]:
# 샘플 데이터
iterator = iter(train_data)
sample_batch = []
for _ in range(8):
    sample_batch.append(next(iterator))

In [13]:
## What happens if we ignore zero-padding

def collate_batch(batch):
    label_list, text_list = [], []
    for (_label, _text) in batch:
        label_list.append(label_pipeline(_label))
        processed_text = torch.tensor(text_pipeline(_text), dtype=torch.int64)
        text_list.append(processed_text)
    label_list = torch.tensor(label_list, dtype=torch.int64)
    text_list = torch.stack(text_list).long()
    return label_list, text_list

In [14]:
# RuntimeError: stack expects each tensor to be equal size, but got [29] at entry 0 and [42] at entry 1
# 텍스트마다 길이가 달라서 텐서로 묶을 수가 없음
collate_batch(sample_batch)

RuntimeError: stack expects each tensor to be equal size, but got [29] at entry 0 and [42] at entry 1

In [15]:
MAX_LEN = 32 
# MAX_LEN 정의
# 1. 하나의 값으로 지정
# 2. 각 배치에서의 최댓값
# 3. 각 배치에서의 최솟값 등등

def collate_batch(batch):
    label_list, text_list = [], []
    for (_label, _text) in batch:
        label_list.append(label_pipeline(_label))
        processed_text = torch.tensor(text_pipeline(_text), dtype=torch.int64)
        # MAX_LEN보다 길면 MAX_LEN 만큼 잘라주기
        if processed_text.size(0) >= MAX_LEN:
            processed_text = processed_text[:MAX_LEN]
        # MAX_LEN보다 짧으면 남은 만큼 0으로 padding
        else:
            processed_text = torch.cat([processed_text, 
                                  torch.zeros(MAX_LEN-processed_text.size(0))])
        text_list.append(processed_text)
    label_list = torch.tensor(label_list, dtype=torch.int64)
    text_list = torch.stack(text_list).long()
    return label_list, text_list

In [16]:
collate_batch(sample_batch)

(tensor([2, 2, 2, 2, 2, 2, 2, 2]),
 tensor([[  431,   425,     1,  1605, 14838,   113,    66,     2,   848,    13,
             27,    14,    27,    15, 50725,     3,   431,   374,    16,     9,
          67507,     6, 52258,     3,    42,  4009,   783,   325,     1,     0,
              0,     0],
         [15874,  1072,   854,  1310,  4250,    13,    27,    14,    27,    15,
            929,   797,   320, 15874,    98,     3, 27657,    28,     5,  4459,
             11,   564, 52790,     8, 80617,  2125,     7,     2,   525,   241,
              3,    28],
         [   58,     8,   347,  4582,   151,    16,   738,    13,    27,    14,
             27,    15,  2384,   452,    92,  2059, 27360,     2,   347,     8,
              2,   738,    11,   271,    42,   240, 51953,    38,     2,   294,
            126,   112],
         [   70,  7376,    58,  1810,    29,   905,   537,  2846,    13,    27,
             14,    27,    15,   838,    39,  4978,    58, 68871,    29,     2,
          

In [17]:
# ----- 예시로 확인하기 -----
# Load dataset
dataloader = DataLoader(train_data, 
                        batch_size=8, 
                        shuffle=False, 
                        collate_fn=collate_batch)

y, x = next(iter(dataloader))
print(x.shape, y.shape) # 첫번째 배치의 데이터 형태 확인

# 파라미터
num_class = len(set([label for (label, text) in train_data]))
vocab_size = len(encoder)
emsize = 64
hidden_dim = 32

# 모델 정의
embedding = nn.Embedding(vocab_size, emsize)
rnn = nn.RNN(emsize,     # input size
             hidden_dim, # hidden size
             1,          # num of layers
             nonlinearity='tanh', 
             bias=True, 
             batch_first=True)
fc = nn.Linear(hidden_dim, num_class)

# 모델 적용
print(x.shape)
h0 = torch.randn(1, 8, hidden_dim) # randomized initial hidden layer
print(h0.shape)
y, h1 = rnn(embedding(x), h0)
print(embedding(x).shape)
print(h1.shape)
print(y.shape)
print(fc(y).shape)
# -----------------

/Users/jeongmoonwon/opt/anaconda3/lib/python3.8/site-packages/torch/utils/data/datapipes/iter/combining.py:248: UserWarning: Some child DataPipes are not exhausted when __iter__ is called. We are resetting the buffer and each child DataPipe will read from the start again.
  warnings.warn("Some child DataPipes are not exhausted when __iter__ is called. We are resetting "


torch.Size([8, 32]) torch.Size([8])
torch.Size([8, 32])
torch.Size([1, 8, 32])
torch.Size([8, 32, 64])
torch.Size([1, 8, 32])
torch.Size([8, 32, 32])
torch.Size([8, 32, 4])


In [18]:
# ----- Model building and implementation -----

In [19]:
class TextClassificationModel(nn.Module):

    def __init__(self, vocab_size, hidden, embed, num_class, batch_size): # (vocab_size, hidden=32, embed=64, num_class=4, batch_size=64)
        super(TextClassificationModel, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed)
        self.rnn = nn.RNN(input_size=embed, 
                          hidden_size=hidden, 
                          num_layers=1, 
                          nonlinearity='tanh', 
                          bias=True, 
                          batch_first=True)
        self.fc = nn.Linear(hidden, num_class)
        self.init_weights()

    def init_weights(self):
        initrange = 0.5
        self.embedding.weight.data.uniform_(-initrange, initrange) # 임베딩 벡터 initialize (-0.5~0.5 사이의 값)
        self.fc.weight.data.uniform_(-initrange, initrange) # linear-regression weight 벡터 initialize (-0.5~0.5 사이의 값)
        self.fc.bias.data.zero_() # linear-regression bias 벡터 initialize (모두 다 0)

    def forward(self, x):
        x = self.embedding(x)
        x, h = self.rnn(x)
        x = torch.mean(x, dim=1) # 마지막 토큰만 가지고 하거나 평균을 내서 fully connected layer 적용 가능
        return self.fc(x)

In [21]:
import time

def train(dataloader):
    model.train()
    total_acc, total_count = 0, 0
    log_interval = 500
    start_time = time.time()

    for idx, (label, text) in enumerate(dataloader):
        label, text = label.to(DEVICE), text.to(DEVICE)
        optimizer.zero_grad()
        predicted_label = model(text)
        loss = criterion(predicted_label, label)
        loss.backward()
        optimizer.step()
        total_acc += (predicted_label.argmax(1) == label).sum().item()
        total_count += label.size(0)
        if idx % log_interval == 0 and idx > 0:
            elapsed = time.time() - start_time
            print('| epoch {:3d} | {:5d}/{:5d} batches '
                  '| accuracy {:8.3f}'.format(epoch, idx, len(dataloader),
                                              total_acc/total_count))
            total_acc, total_count = 0, 0
            start_time = time.time()

            
def evaluate(dataloader):
    model.eval()
    total_acc, total_count = 0, 0

    with torch.no_grad():
        for idx, (label, text) in enumerate(dataloader):
            label, text = label.to(DEVICE), text.to(DEVICE)
            predicted_label = model(text)
            loss = criterion(predicted_label, label)
            total_acc += (predicted_label.argmax(1) == label).sum().item()
            total_count += label.size(0)
    return total_acc/total_count

In [22]:
from torchtext.data.functional import to_map_style_dataset

EPOCHS = 5
LR = 1
BATCH_SIZE = 64
num_class = len(set([label for (label, text) in train_data]))
vocab_size = len(encoder)
emsize = 64
hidden_dim = 32

model = TextClassificationModel(vocab_size, hidden_dim, emsize, num_class, BATCH_SIZE).to(DEVICE)

criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, 1.0, gamma=0.1)
total_accu = None

train_dataset = to_map_style_dataset(train_data)
test_dataset = to_map_style_dataset(test_data)
train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                              shuffle=True, collate_fn=collate_batch)
valid_dataloader = DataLoader(test_dataset, batch_size=BATCH_SIZE,
                              shuffle=True, collate_fn=collate_batch)

In [ ]:
## Train the RNN for text classification

for epoch in range(1, EPOCHS + 1):
    epoch_start_time = time.time()
    train(train_dataloader)
    accu_val = evaluate(valid_dataloader)
    if total_accu is not None and total_accu > accu_val:
        scheduler.step()
    else:
        total_accu = accu_val
    print('-' * 59)
    print('| end of epoch {:3d} | time: {:5.2f}s | '
          'valid accuracy {:8.3f} '.format(epoch,
                                           time.time() - epoch_start_time,
                                           accu_val))
    print('-' * 59)